Single allele groupped structue models.

In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/alleles_with_groups_single_allele_imbalanced_data/" #!CHANGE for bwv!

In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-DRB1*01:01',
 'HLA-A*03:01',
 'HLA-B*27:05',
 'HLA-B*07:02',
 'HLA-A*01:01',
 'HLA-A*02:01']

## 2.0 Merge alleles together

In [4]:
# combination_size = len(ALLELES)
ALLELES = ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01']
combination_size = 1
TOTAL_SPLITS = 5

In [5]:
import itertools
from collections import defaultdict
per_allele_per_kfold_per_length_binders_train_new = dict()

per_allele_per_kfold_per_length_binders_test_new = dict()

per_kfold_per_length_per_allele_counts_old_train = dict()
per_kfold_per_length_per_allele_counts_old_test = dict()





OLD_ALLELES = ALLELES
NEW_ALELLES = list()
selected_lengths = np.arange(8,20)

# initialize count dicts
for allele_name in OLD_ALLELES:
    for split_num in range(TOTAL_SPLITS):
        per_kfold_per_length_per_allele_counts_old_train[split_num] = defaultdict(int)
        per_kfold_per_length_per_allele_counts_old_test[split_num] = defaultdict(int)
        
    



for combination in itertools.combinations(ALLELES, combination_size):
    allele_name = "+".join(combination)
    NEW_ALELLES.append(allele_name)
    print(allele_name)
    per_allele_per_kfold_per_length_binders_train_new[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_new[allele_name] = dict()


    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num] = dict()

        list_of_length_sets = list()
        for old_allele in combination:
            lengths = set(per_allele_per_kfold_per_length_binders_train[old_allele][split_num].keys())
            list_of_length_sets.append(lengths)
        intersected_lengths = set.intersection(*list_of_length_sets)

        # #calculate allele_weights here
        # # allele weight is   len(allele data)/ total_data_length
        # for old_allele in combination:
        #     lengths_list = {
        #         length: len(length_binders)
        #         for length, length_binders
        #         in per_allele_per_kfold_per_length_binders_train[old_allele][split_num].values()
        #     }


        for target_length in intersected_lengths:
            data_arrays_train = list()
            data_arrays_test = list()
            if target_length not in per_kfold_per_length_per_allele_counts_old_train[split_num]:
                per_kfold_per_length_per_allele_counts_old_train[split_num][target_length] = dict()
                per_kfold_per_length_per_allele_counts_old_test[split_num][target_length] = dict()
            if split_num == 0:
                print(f"Length {target_length}:" , end=" ")
            for old_allele in combination:
                single_allele_data_array_train = per_allele_per_kfold_per_length_binders_train[old_allele][split_num][target_length]

                per_kfold_per_length_per_allele_counts_old_train[split_num][target_length][old_allele] = len(single_allele_data_array_train)
                if split_num == 0:
                    print(f"{old_allele} - {len(single_allele_data_array_train)}", end= " ")
                data_arrays_train.append(single_allele_data_array_train)
                single_allele_data_array_test = per_allele_per_kfold_per_length_binders_test[old_allele][split_num][target_length]
                per_kfold_per_length_per_allele_counts_old_test[split_num][target_length][old_allele] = len(single_allele_data_array_test)
                if split_num == 0:
                    # print(f"{old_allele} - {len(single_allele_data_array_test)}", end= " ")
                    pass
                data_arrays_test.append(single_allele_data_array_test)
            if split_num == 0:
                print()

            per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_train)
            per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_test)


per_allele_per_kfold_per_length_binders_train = per_allele_per_kfold_per_length_binders_train_new
per_allele_per_kfold_per_length_binders_test = per_allele_per_kfold_per_length_binders_test_new
ALLELES = NEW_ALELLES

HLA-A*01:01
Length 8: HLA-A*01:01 - 764 
Length 9: HLA-A*01:01 - 5052 
Length 10: HLA-A*01:01 - 2464 
Length 11: HLA-A*01:01 - 1430 
Length 12: HLA-A*01:01 - 1106 
Length 13: HLA-A*01:01 - 842 
Length 14: HLA-A*01:01 - 568 
Length 15: HLA-A*01:01 - 464 
Length 16: HLA-A*01:01 - 345 
Length 17: HLA-A*01:01 - 209 
Length 18: HLA-A*01:01 - 128 
Length 19: HLA-A*01:01 - 75 
Length 20: HLA-A*01:01 - 54 
Length 21: HLA-A*01:01 - 29 
Length 22: HLA-A*01:01 - 21 
Length 23: HLA-A*01:01 - 17 
Length 24: HLA-A*01:01 - 16 
Length 25: HLA-A*01:01 - 8 
Length 26: HLA-A*01:01 - 7 
Length 28: HLA-A*01:01 - 6 
Length 29: HLA-A*01:01 - 4 
Length 30: HLA-A*01:01 - 5 
HLA-A*03:01
Length 8: HLA-A*03:01 - 93 
Length 9: HLA-A*03:01 - 5132 
Length 10: HLA-A*03:01 - 2184 
Length 11: HLA-A*03:01 - 691 
Length 12: HLA-A*03:01 - 202 
Length 13: HLA-A*03:01 - 98 
Length 14: HLA-A*03:01 - 31 
Length 15: HLA-A*03:01 - 18 
Length 16: HLA-A*03:01 - 12 
Length 17: HLA-A*03:01 - 5 
Length 18: HLA-A*03:01 - 4 
HLA-A*02:

## Calculate weights for data

In [6]:

def extract_count_df(per_kfold_per_length_per_allele_counts):
    result_dict = {}
    for kfold,  per_length_per_allele_counts in per_kfold_per_length_per_allele_counts.items():
        for length, per_allele_counts in per_length_per_allele_counts.items():
            for allele_name , count in  per_allele_counts.items():
                result_dict[(allele_name, kfold, length)] =  {'cnt': count }
    count_df = pd.DataFrame.from_dict(result_dict, orient="index")
    count_df.index.names = ['allele', 'split', 'length']
    # count_df = count_df.groupby(['allele', 'length'])['cnt'].mean()
    return count_df



count_df_train = extract_count_df(per_kfold_per_length_per_allele_counts_old_train)
count_df_test = extract_count_df(per_kfold_per_length_per_allele_counts_old_test)


In [7]:
count_df_train.index.levels

FrozenList([['HLA-A*01:01', 'HLA-A*02:01', 'HLA-A*03:01'], [0, 1, 2, 3, 4], [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]])

# 2. Train HMMS
## 2.1 Code for HMM
### 2.1.1 Training code

In [8]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions, init_grouped_structure_model

TARGET_LENGTHS= [9, 10]
NUM_RUNS = 2
TOTAL_SPLITS = 5
USE_WEIGHTS = False

### Define weights for data

In [9]:
per_allele_per_kfold_per_length_weights = dict()


for combined_allele_name in ALLELES:
    per_allele_per_kfold_per_length_weights[combined_allele_name] = dict()
    old_alleles = combined_allele_name.split('+')
    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_weights[combined_allele_name][split_num] = dict()
        sub_count_df_train = count_df_train.loc[(count_df_train.index.get_level_values('split') == 0) &
                                                (count_df_train.index.get_level_values('allele').isin(old_alleles)) &
                                                (count_df_train.index.get_level_values('length').isin(TARGET_LENGTHS))]
        total_data_length = sub_count_df_train['cnt'].sum()
        print(f"Total data length for split {split_num} is {total_data_length}")
        for target_length in TARGET_LENGTHS:

            current_weights_list = list()
            weights_dict = dict()
            for old_allele_name in old_alleles:
                old_allele_cnt = count_df_train.loc[(old_allele_name, split_num, target_length)]['cnt']
                current_weights_list.extend([total_data_length/old_allele_cnt if USE_WEIGHTS else 1 for i in range(old_allele_cnt)])
                weights_dict[old_allele_name] = total_data_length/old_allele_cnt if USE_WEIGHTS else 1
            per_allele_per_kfold_per_length_weights[combined_allele_name][split_num][target_length] = current_weights_list
            print(f"Weights for lentgh {target_length} are {weights_dict}")


Total data length for split 0 is 7516
Weights for lentgh 9 are {'HLA-A*01:01': 1}
Weights for lentgh 10 are {'HLA-A*01:01': 1}
Total data length for split 1 is 7516
Weights for lentgh 9 are {'HLA-A*01:01': 1}
Weights for lentgh 10 are {'HLA-A*01:01': 1}
Total data length for split 2 is 7516
Weights for lentgh 9 are {'HLA-A*01:01': 1}
Weights for lentgh 10 are {'HLA-A*01:01': 1}
Total data length for split 3 is 7516
Weights for lentgh 9 are {'HLA-A*01:01': 1}
Weights for lentgh 10 are {'HLA-A*01:01': 1}
Total data length for split 4 is 7516
Weights for lentgh 9 are {'HLA-A*01:01': 1}
Weights for lentgh 10 are {'HLA-A*01:01': 1}
Total data length for split 0 is 7316
Weights for lentgh 9 are {'HLA-A*03:01': 1}
Weights for lentgh 10 are {'HLA-A*03:01': 1}
Total data length for split 1 is 7316
Weights for lentgh 9 are {'HLA-A*03:01': 1}
Weights for lentgh 10 are {'HLA-A*03:01': 1}
Total data length for split 2 is 7316
Weights for lentgh 9 are {'HLA-A*03:01': 1}
Weights for lentgh 10 are {'H

## 2.2 Train
### 2.2.1 Define treshold

In [10]:
stop_threshold = 0
cutoff = 0.0001
per_symbol_portion = -50 / 9
for target_length in TARGET_LENGTHS:
    target_length_for_training = target_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

19.474444444444444

### 2.2.2 Prepare models

In [11]:
# Training params
algorithm = 'viterbi'

params = dict()


params['st'] = 25
params['multiple_check_input'] = False
params['alg'] = algorithm
params['min_iters'] = 5
params['stop_threshold'] = stop_threshold
params['emission_pseudocount'] = 1
params['transition_pseudocount'] = 0
params['verbose'] = True
params['lr_decay'] = 0
params['edge_inertia'] = 0.25
params['distribution_inertia'] = 0
params['groups'] = 5

assert params['st'] % params['groups'] == 0

params_for_string = {key: value for key, value in params.items() if key in ['st', 'alg', 'min_iters']}
params_string = "-".join([f"{value}-{key}" for key, value in params_for_string.items()])
lengths_string = "-".join(map(str, TARGET_LENGTHS))


In [12]:
model = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])

Initialize grouped structure model


In [13]:
model.dense_transition_matrix()

array([[0.        , 0.19047619, 0.19047619, 0.19047619, 0.19047619,
        0.04761905, 0.04761905, 0.04761905, 0.04761905, 0.04761905,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.19047619, 0.        , 0.19047619, 0.19047619, 0.19047619,
        0.04761905, 0.04761905, 0.04761905, 0.04761905, 0.04761905,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.19047619, 0.19047619, 0.        , 0.19047619, 0.19047619,
        0.04761905, 0.04761905, 0.04761905, 0.04761905, 0.04761905,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.    

In [14]:
per_allele_per_run_per_split_prepared_models = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        for split_num in range(TOTAL_SPLITS):
            prepared_model  = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])

            prepared_model.name = f'{run_num}_run-{lengths_string}-{params_string}_model-{target_allele_name}-{split_num}'
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped structure model
Initialize grouped s

In [15]:
ALLELES

['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01']

In [16]:
%%time


per_name_models = dict()
per_allele_per_run_per_split_new_histories = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"run {run_num}")
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        per_kfold_per_length_weights = per_allele_per_kfold_per_length_weights[allele_name]


        models, histories = train_model(allele_name=allele_name,
                                        per_kfold_per_length_data=per_kfold_per_length_data,
                                        per_kfold_per_length_weights_for_data=per_kfold_per_length_weights,
                                        target_lengths=TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm=params['alg'],
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount=params['transition_pseudocount'],
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia=params['distribution_inertia'],
                                        lr_decay=params['lr_decay'])
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


run 0
split 0
train size 7516
Shuffling for fairness 
Simple training: Train Length 7516, init_batch_size None
[1] Improvement: -127911.0026118265	Time (s): 1.199
[2] Improvement: 3780.050843114237	Time (s): 1.391
[3] Improvement: 1907.8321196908946	Time (s): 1.019
[4] Improvement: 1481.3511138705653	Time (s): 1.366
[5] Improvement: 1027.568309585913	Time (s): 1.46
[6] Improvement: 605.2730667700816	Time (s): 1.908
[7] Improvement: 539.0742323677114	Time (s): 1.428
[8] Improvement: 457.7253969939484	Time (s): 1.221
[9] Improvement: 311.59827992419014	Time (s): 1.194
[10] Improvement: 190.64243157554301	Time (s): 1.209
[11] Improvement: 198.4289257141936	Time (s): 1.497
[12] Improvement: 178.8887214648712	Time (s): 1.677
[13] Improvement: 140.63517017051345	Time (s): 1.506
[14] Improvement: 134.41573493304895	Time (s): 1.314
[15] Improvement: 73.60155823364039	Time (s): 1.125
[16] Improvement: 62.984140389133245	Time (s): 1.093
[17] Improvement: 50.56279205446481	Time (s): 1.256
[18] Im

KeyboardInterrupt: 

In [17]:
from hmm_visualization_methods import *

In [18]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
Wall time: 1.2 s


In [19]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
Wall time: 16.4 s


In [20]:
from hmm_visualization_methods import *
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=2)

Converting graph for model (0_run-9-10-25-st-viterbi-alg-5-min_iters_model-HLA_A_01_01-0) (Delete zero edges)


## Aux viterbi

In [21]:
per_allele_per_run_per_split_new_histories = dict()

for allele_name in ALLELES:
    for run_num in range(NUM_RUNS):
        for split_num, model in per_allele_per_run_per_split_prepared_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_plus_viterbi"

            for state in prepared_model.states:
                if state is not prepared_model.start and state is not prepared_model.end:
                    state.distribution.thaw()
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

In [22]:
per_name_new_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        per_kfold_per_length_weights = per_allele_per_kfold_per_length_weights[allele_name]
        models, histories = train_model(allele_name=allele_name,
                                        per_kfold_per_length_data=per_kfold_per_length_data,
                                        per_kfold_per_length_weights_for_data=per_kfold_per_length_weights,
                                        target_lengths=TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm='viterbi',
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount= 1,  #!CHANGE for bwv!
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia= params['distribution_inertia'],
                                        lr_decay=params['lr_decay']
                                        )
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_new_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

split 0
train size 7516
Shuffling for fairness 
Simple training: Train Length 7516, init_batch_size None
[1] Improvement: 40.682193915039534	Time (s): 1.391
[2] Improvement: 18.686458945361665	Time (s): 1.142
[3] Improvement: 13.0001256922842	Time (s): 1.231
[4] Improvement: 15.058430018485524	Time (s): 1.567
[5] Improvement: 16.28664075443521	Time (s): 1.586
Total Training Improvement: 103.71384932560613
Total Training Time (s): 9.0560
split 0
train size 7516
Shuffling for fairness 
Simple training: Train Length 7516, init_batch_size None
[1] Improvement: 27304.567264577723	Time (s): 1.109
[2] Improvement: 3192.785621067276	Time (s): 1.134
[3] Improvement: 1246.9337580135034	Time (s): 1.161
[4] Improvement: 992.6458092281537	Time (s): 1.095
[5] Improvement: 940.6375258080079	Time (s): 2.15
[6] Improvement: 634.3625175153429	Time (s): 3.252
[7] Improvement: 456.5312928430212	Time (s): 2.526
[8] Improvement: 460.6377893036115	Time (s): 2.345
[9] Improvement: 413.6806503486587	Time (s): 

KeyboardInterrupt: 

In [ ]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

In [ ]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

In [ ]:
per_name_pyviz = dict()
for name, model in per_name_new_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)